# Balanced Benchmark Across Federated Heterogeneity Types

**Purpose.** Run a controlled comparison of FedAvg, FedProx, and
SCAFFOLD under label, quantity, and feature heterogeneity.

The experiment fixes the model, optimizer, communication budget,
client count, data splits, and initial weights within each
condition. The only intended changes are the heterogeneity
condition, random seed, and federated algorithm.

## Research questions

1. Does algorithm performance depend on the *type* and *severity*
   of heterogeneity?
2. Are observed differences robust across seeds 42, 43, and 44?
3. Do aggregate accuracy, worst-client accuracy, convergence, and
   runtime tell the same story?

This notebook executes and validates the protocol. Statistical
analysis and figures are separated into Notebook 10.


## 1. Reproducible source snapshot

The benchmark is tied to an immutable Git commit. Pinning the
commit prevents later source changes from silently altering the
experiment.


In [2]:
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = (
    "https://github.com/khanhhuyenpham/"
    "federated-learning-under-heterogeneity.git"
)

BRANCH = "main"
COMMIT = "fd2b045544297350d3bbe4a44a81f1a49392d2cb"

REPOSITORY_DIRECTORY = Path(
    "/kaggle/working/"
    "federated-learning-under-heterogeneity"
)

if not REPOSITORY_DIRECTORY.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            BRANCH,
            REPOSITORY_URL,
            str(REPOSITORY_DIRECTORY),
        ],
        check=True,
    )

subprocess.run(
    [
        "git",
        "fetch",
        "origin",
        BRANCH,
    ],
    cwd=REPOSITORY_DIRECTORY,
    check=True,
)

subprocess.run(
    [
        "git",
        "checkout",
        COMMIT,
    ],
    cwd=REPOSITORY_DIRECTORY,
    check=True,
)

sys.path.insert(
    0,
    str(REPOSITORY_DIRECTORY),
)

print("Repository:", REPOSITORY_DIRECTORY)
print("Commit:", COMMIT)


Cloning into '/kaggle/working/federated-learning-under-heterogeneity'...


Repository: /kaggle/working/federated-learning-under-heterogeneity
Commit: fd2b045544297350d3bbe4a44a81f1a49392d2cb


From https://github.com/khanhhuyenpham/federated-learning-under-heterogeneity
 * branch            main       -> FETCH_HEAD
Note: switching to 'fd2b045544297350d3bbe4a44a81f1a49392d2cb'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at fd2b045 feat: define balanced benchmark protocol


## 2. Compute environment

Record the Python, PyTorch, CUDA, and GPU versions. The full study
requires a CUDA device and was designed for Kaggle GPU execution.


In [3]:
import platform
import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )

assert torch.cuda.is_available()


Python: 3.12.13
PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4


## 3. Quality gate

The complete test suite must pass before any expensive training.
This protects the run contract, adapters, storage layer, condition
construction, and algorithm invariants.


In [4]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests",
        "-q",
    ],
    cwd=REPOSITORY_DIRECTORY,
    check=True,
)


........................................................................ [ 26%]
........................................................................ [ 53%]
........................................................................ [ 80%]
....................................................                     [100%]
=============================== warnings summary ===============================
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64
  /usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
    prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))

../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python

CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', 'tests', '-q'], returncode=0)

## 4. Imports


In [5]:
from dataclasses import asdict, replace
from pathlib import Path
import json
import shutil

import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from src.benchmark_protocol import (
    MAIN_BENCHMARK_SEEDS,
    SELECTED_FEDPROX_MU,
    build_main_benchmark_algorithms,
    build_main_benchmark_conditions,
    make_training_config,
)
from src.benchmark_storage import (
    BenchmarkRunStore,
)
from src.benchmark_suite import (
    run_benchmark_suite,
)
from src.condition_artifacts import (
    build_condition_artifacts,
)
from src.models import SimpleMLP


## 5. Dataset, evaluation set, and model

MNIST's 60,000 training examples are partitioned among five
clients. Each client's partition is split into 80% training, 10%
validation, and 10% local test data. The official 10,000-example
MNIST test set remains a clean global evaluation set.

All methods use `SimpleMLP(784 → 128 → 10)`, cross-entropy loss,
full client participation, 20 communication rounds, one local
epoch, learning rate 0.01, and batch size 64.


In [6]:
device = torch.device("cuda")

transform = transforms.ToTensor()

train_ds = datasets.MNIST(
    root="/kaggle/working/data",
    train=True,
    download=True,
    transform=transform,
)

test_ds = datasets.MNIST(
    root="/kaggle/working/data",
    train=False,
    download=True,
    transform=transform,
)

global_test_loader = DataLoader(
    test_ds,
    batch_size=256,
    shuffle=False,
)


def model_factory():
    return SimpleMLP(
        num_inputs=28 * 28,
        num_classes=10,
    )


100%|██████████| 9.91M/9.91M [00:00<00:00, 17.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 440kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.64MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.37MB/s]


## 6. Prespecified benchmark matrix

Seven conditions are evaluated for each of three seeds:

| Family | Conditions |
|---|---|
| Reference | IID |
| Label skew | Dirichlet α = 0.5 and 0.1 |
| Quantity skew | Dirichlet α = 0.5 and 0.1, minimum 100 samples/client |
| Rotation shift | maximum absolute rotation 10° and 20° |

Each condition is paired with FedAvg (μ=0), FedProx using the
independently selected pilot value μ=0.01, and SCAFFOLD. This gives
7 × 3 × 3 = 63 unique runs.


In [7]:
condition_configs = (
    build_main_benchmark_conditions()
)

algorithm_specs = (
    build_main_benchmark_algorithms()
)

assert len(condition_configs) == 21
assert len(algorithm_specs) == 3

assert {
    specification.algorithm
    for specification in algorithm_specs
} == {
    "fedavg",
    "fedprox",
    "scaffold",
}

expected_runs = len(condition_configs) * len(algorithm_specs)
assert expected_runs == 63

print("Seeds:", MAIN_BENCHMARK_SEEDS)
print("Selected FedProx mu:", SELECTED_FEDPROX_MU)
print("Conditions:", len(condition_configs))
print("Algorithms:", len(algorithm_specs))
print("Expected runs:", expected_runs)


Seeds: (42, 43, 44)
Selected FedProx mu: 0.01
Conditions: 21
Algorithms: 3
Expected runs: 63


## 7. Condition preflight

Construct every partition before training and record its realized
characteristics. `mean_label_tvd`, `quantity_cv`, and
`rotation_std_degrees` verify that the requested conditions produce
meaningfully different forms of heterogeneity. This step also
catches empty or malformed client splits before GPU time is spent.


In [8]:
preflight_rows = []

for condition in condition_configs:
    training_config = make_training_config(
        condition
    )

    artifacts = build_condition_artifacts(
        train_ds=train_ds,
        config=condition,
        batch_size=training_config.batch_size,
        train_fraction=0.8,
        validation_fraction=0.1,
    )

    preflight_rows.append(
        asdict(artifacts.metadata)
    )

condition_preflight = pd.DataFrame(
    preflight_rows
)

assert len(condition_preflight) == 21

preflight_view = (
    condition_preflight
    .sort_values(
        ["seed", "mode", "alpha", "max_abs_angle"],
        na_position="first",
    )
    [
        [
            "seed", "mode", "alpha",
            "min_samples_per_client", "max_abs_angle",
            "min_client_size", "max_client_size",
            "quantity_cv", "mean_label_tvd",
            "rotation_std_degrees",
        ]
    ]
)

assert condition_preflight["dataset_size"].eq(len(train_ds)).all()
assert condition_preflight["num_clients"].eq(5).all()
display(preflight_view)


,seed,mode,alpha,min_samples_per_client,max_abs_angle,min_client_size,max_client_size,quantity_cv,mean_label_tvd,rotation_std_degrees
0,42,iid,NaN,1,0.0,12000,12000,0.000000,0.008903,0.000000
2,42,label_skew,0.1,1,0.0,12000,12000,0.000000,0.449407,0.000000
1,42,label_skew,0.5,1,0.0,12000,12000,0.000000,0.318933,0.000000
4,42,quantity_skew,0.1,100,0.0,100,36140,1.097790,0.049099,0.000000
3,42,quantity_skew,0.5,100,0.0,296,26738,0.852848,0.031760,0.000000
5,42,rotation_shift,NaN,1,10.0,12000,12000,0.000000,0.008903,7.071068
6,42,rotation_shift,NaN,1,20.0,12000,12000,0.000000,0.008903,14.142136
7,43,iid,NaN,1,0.0,12000,12000,0.000000,0.010400,0.000000
9,43,label_skew,0.1,1,0.0,12000,12000,0.000000,0.546450,0.000000
8,43,label_skew,0.5,1,0.0,12000,12000,0.000000,0.415937,0.000000


## 8. One-round smoke test

Use the most severe rotation condition to exercise transformed
data, all three algorithms, both evaluation paths, and checkpoint
storage. Smoke-test artifacts use a separate directory and can
never be mistaken for main-study results.

The completed main archive has already passed this gate, so the
smoke test is disabled by default during result restoration.


In [9]:
RUN_SMOKE_TEST = False

smoke_condition = next(
    condition
    for condition in condition_configs
    if (
        condition.seed == 42
        and condition.mode == "rotation_shift"
        and condition.max_abs_angle == 20.0
    )
)


def smoke_training_config_factory(condition):
    return replace(
        make_training_config(condition),
        num_rounds=1,
    )


smoke_store = BenchmarkRunStore(
    "/kaggle/working/results/"
    "balanced_benchmark_smoke_v1"
)

print("Run smoke test:", RUN_SMOKE_TEST)


Run smoke test: False


In [10]:
if RUN_SMOKE_TEST:
    smoke_result = run_benchmark_suite(
        train_ds=train_ds,
        global_test_loader=global_test_loader,
        condition_configs=[smoke_condition],
        algorithm_specs=algorithm_specs,
        training_config_factory=(
            smoke_training_config_factory
        ),
        model_factory=model_factory,
        loss_fn=nn.CrossEntropyLoss(),
        device=device,
        train_fraction=0.8,
        validation_fraction=0.1,
        verbose=False,
        is_run_complete=smoke_store.is_spec_complete,
        on_run_complete=smoke_store.save_spec_result,
    )

    assert len(smoke_store.load_completed_summary()) == 3
    print("Smoke test passed.")
else:
    print(
        "Smoke test skipped: using the completed and "
        "previously validated benchmark archive."
    )


Smoke test skipped: using the completed and previously validated benchmark archive.


## 9. Resumable main execution

A run is considered complete only after its completion marker is
written. Each finished result is saved immediately, and a ZIP
checkpoint is refreshed after every three runs. Restarting this
cell skips completed keys instead of retraining them.


In [11]:
OUTPUT_DIRECTORY = Path(
    "/kaggle/working/results/"
    "balanced_benchmark_v1"
)

store = BenchmarkRunStore(
    OUTPUT_DIRECTORY
)


### Restore the completed benchmark

Kaggle may expose the uploaded artifact either as a literal ZIP or
as an already-extracted dataset containing `runs/` and `combined/`.
The next cell supports both representations, copies the read-only
input into `/kaggle/working`, and requires all 63 completion markers.


In [13]:
import zipfile

KAGGLE_INPUT_ROOT = Path("/kaggle/input")

if not KAGGLE_INPUT_ROOT.exists():
    raise RuntimeError(
        "Kaggle input directory is unavailable. Attach the "
        "balanced_benchmark_v1 dataset before continuing."
    )

OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

existing_markers = list(
    OUTPUT_DIRECTORY.glob("runs/*/COMPLETED.json")
)

if len(existing_markers) == expected_runs:
    print(
        "Using an already-restored working directory:",
        OUTPUT_DIRECTORY,
    )
else:
    extracted_roots = []

    for runs_directory in KAGGLE_INPUT_ROOT.rglob("runs"):
        if not runs_directory.is_dir():
            continue

        marker_count = len(
            list(runs_directory.glob("*/COMPLETED.json"))
        )

        if marker_count == expected_runs:
            extracted_roots.append(runs_directory.parent)

    archive_matches = list(
        KAGGLE_INPUT_ROOT.rglob(
            "balanced_benchmark_v1_complete.zip"
        )
    )

    if len(extracted_roots) == 1:
        input_result_root = extracted_roots[0]
        print(
            "Using extracted Kaggle input:",
            input_result_root,
        )
        shutil.copytree(
            input_result_root,
            OUTPUT_DIRECTORY,
            dirs_exist_ok=True,
        )

    elif len(archive_matches) == 1:
        benchmark_archive = archive_matches[0]
        print("Using ZIP archive:", benchmark_archive)

        with zipfile.ZipFile(benchmark_archive, "r") as archive:
            corrupt_member = archive.testzip()

            if corrupt_member is not None:
                raise RuntimeError(
                    "Corrupt archive member: "
                    f"{corrupt_member}"
                )

            archive.extractall(OUTPUT_DIRECTORY)

    else:
        raise RuntimeError(
            "Could not identify exactly one complete benchmark "
            "input. "
            f"Extracted roots: {extracted_roots}. "
            f"ZIP archives: {archive_matches}."
        )

completion_markers = list(
    OUTPUT_DIRECTORY.glob("runs/*/COMPLETED.json")
)
assert len(completion_markers) == expected_runs

restored_summary = store.load_completed_summary()
assert len(restored_summary) == expected_runs
assert set(restored_summary["algorithm"]) == {
    "fedavg",
    "fedprox",
    "scaffold",
}
assert set(restored_summary["seed"]) == set(MAIN_BENCHMARK_SEEDS)

print("Restored completion markers:", len(completion_markers))
print("Restored summary rows:", len(restored_summary))

display(
    restored_summary
    .groupby("algorithm")
    .agg(
        runs=("seed", "size"),
        mean_global_accuracy=("global_test_accuracy", "mean"),
    )
    .round(4)
)


Using extracted Kaggle input: /kaggle/input/datasets/harleypham257/balanced-benchmark-v1
Restored completion markers: 63
Restored summary rows: 63


,runs,mean_global_accuracy
algorithm,,
fedavg,21,0.9011
fedprox,21,0.9009
scaffold,21,0.9079


### No-retraining safety gate

Every requested condition–algorithm key must already be complete.
If even one key is missing, the notebook stops before entering the
suite rather than silently launching new training.


In [14]:
incomplete_specs = []

for condition in condition_configs:
    for specification in algorithm_specs:
        if not store.is_spec_complete(condition, specification):
            incomplete_specs.append((condition, specification))

if incomplete_specs:
    raise RuntimeError(
        "Refusing to start the main suite because some runs "
        "would be retrained: "
        f"{incomplete_specs}"
    )

assert len(completion_markers) == expected_runs
assert len(restored_summary) == expected_runs

print("All 63 requested runs are complete.")
print("The main suite will perform no training.")


All 63 requested runs are complete.
The main suite will perform no training.


In [15]:
def save_main_checkpoint(
    condition,
    specification,
    result,
):
    store.save_spec_result(
        condition,
        specification,
        result,
    )

    completed_summary = (
        store.load_completed_summary()
    )

    completed_count = len(
        completed_summary
    )

    print(
        f"Saved {completed_count}/63:",
        condition.mode,
        condition.seed,
        specification.algorithm,
        specification.mu,
    )

    if (
        completed_count % 3 == 0
        or completed_count == 63
    ):
        shutil.make_archive(
            "/kaggle/working/"
            "balanced_benchmark_v1_checkpoint",
            "zip",
            root_dir=OUTPUT_DIRECTORY,
        )


In [16]:
main_result = run_benchmark_suite(
    train_ds=train_ds,
    global_test_loader=global_test_loader,
    condition_configs=condition_configs,
    algorithm_specs=algorithm_specs,
    training_config_factory=(
        make_training_config
    ),
    model_factory=model_factory,
    loss_fn=nn.CrossEntropyLoss(),
    device=device,
    train_fraction=0.8,
    validation_fraction=0.1,
    verbose=False,
    is_run_complete=(
        store.is_spec_complete
    ),
    on_run_complete=(
        save_main_checkpoint
    ),
)


## 10. Coverage and integrity checks

The study is accepted only if all 63 unique condition–seed–method
combinations are present and each condition contains all three
algorithms.


In [17]:
combined_summary = store.load_completed_summary()

assert len(combined_summary) == expected_runs
assert set(combined_summary["seed"]) == set(MAIN_BENCHMARK_SEEDS)
assert set(combined_summary["algorithm"]) == {
    "fedavg",
    "fedprox",
    "scaffold",
}

coverage = (
    combined_summary
    .groupby(
        [
            "mode",
            "seed",
            "alpha",
            "min_samples_per_client",
            "max_abs_angle",
        ],
        dropna=False,
    )["algorithm"]
    .nunique()
)

assert len(coverage) == len(condition_configs)
assert coverage.eq(len(algorithm_specs)).all()

display(
    combined_summary
    .groupby("algorithm")
    .agg(
        runs=("seed", "size"),
        mean_global_accuracy=("global_test_accuracy", "mean"),
        mean_runtime_seconds=("runtime_seconds", "mean"),
    )
    .round(4)
)


,runs,mean_global_accuracy,mean_runtime_seconds
algorithm,,,
fedavg,21,0.9011,222.9305
fedprox,21,0.9009,223.4623
scaffold,21,0.9079,226.4235


## 11. Export analysis-ready tables

Row-count assertions encode the 20-round, five-client,
full-participation protocol. The combined tables are the only
inputs required by Notebook 10; model and method-specific states
remain available inside each run directory for auditability.


In [18]:
combined_directory = OUTPUT_DIRECTORY / "combined"
combined_directory.mkdir(parents=True, exist_ok=True)

combined_summary.to_csv(
    combined_directory / "summary.csv",
    index=False,
)

expected_rows = {
    "validation_by_round": 63 * 21,
    "validation_by_client": 63 * 21 * 5,
    "client_update_diagnostics": 63 * 20 * 5,
    "pairwise_update_diagnostics": 63 * 20 * 10,
    "round_diagnostics": 63 * 20,
    "local_test_by_client": 63 * 5,
}

for table_name, expected_count in expected_rows.items():
    table = store.load_completed_table(table_name)
    assert len(table) == expected_count, (
        table_name,
        len(table),
        expected_count,
    )
    table.to_csv(
        combined_directory / f"{table_name}.csv",
        index=False,
    )

condition_preflight.to_csv(
    combined_directory / "condition_preflight.csv",
    index=False,
)

print("Validated and exported:", combined_directory)


/kaggle/working/federated-learning-under-heterogeneity/src/benchmark_storage.py:663: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(
/kaggle/working/federated-learning-under-heterogeneity/src/benchmark_storage.py:663: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(
/kaggle/working/federated-learning-under-heterogeneity/src/benchmark_storage.py:663: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future versio

Validated and exported: /kaggle/working/results/balanced_benchmark_v1/combined


/kaggle/working/federated-learning-under-heterogeneity/src/benchmark_storage.py:663: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(


## 12. Package the reproducibility artifact

The final ZIP contains the combined tables plus every run's model
state, metadata, diagnostics, summaries, and completion marker.


In [19]:
final_archive = shutil.make_archive(
    "/kaggle/working/"
    "balanced_benchmark_v1_complete",
    "zip",
    root_dir=OUTPUT_DIRECTORY,
)

print(final_archive)


/kaggle/working/balanced_benchmark_v1_complete.zip


## Handoff

The benchmark execution is complete when the archive is created
and all integrity checks pass. Notebook 10 treats this archive as
immutable evidence: it performs no training and reports only
prespecified aggregate, robustness, fairness, convergence, and
runtime analyses.
